# Breast Cancer Classification: Decision Tree vs Random Forest

A short comparison exercise on scikit-learn's built-in breast cancer dataset. The question was simple: does an ensemble of trees actually outperform a single decision tree on this dataset, and by how much? Both models are evaluated with 5-fold cross-validation rather than a single train/test split, which was a deliberate choice to get a more stable accuracy estimate than one split would give.


In [ ]:
import numpy as np
from sklearn.model_selection import KFold
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import confusion_matrix


## Loading the data


In [ ]:
X, y = load_breast_cancer(return_X_y=True, as_frame=False)
print(X.shape, y.shape)


## Decision tree baseline

A single decision tree, evaluated across 5 folds.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

dt = tree.DecisionTreeClassifier(random_state=10)
kf = KFold(n_splits=5, shuffle=True, random_state=10)

dt_accuracy = 0
for k, (train_indices, test_indices) in enumerate(kf.split(X)):
  dt.fit(X[train_indices], y[train_indices])
  fold_score = dt.score(X[test_indices], y[test_indices]) * 100
  dt_accuracy += fold_score
  print("[fold {0}] score: {1:.5f}".format(k, fold_score))

print('Mean DT accuracy:', dt_accuracy / 5)


## Random forest comparison

An ensemble of 100 trees (scikit-learn's default `n_estimators`), same cross-validation setup, same random seed on the fold split so the comparison is apples to apples.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=10)
kf = KFold(n_splits=5, shuffle=True, random_state=10)

rf_accuracy = 0
for k, (train_indices, test_indices) in enumerate(kf.split(X)):
  rf.fit(X[train_indices], y[train_indices])
  fold_score = rf.score(X[test_indices], y[test_indices]) * 100
  rf_accuracy += fold_score
  print("[fold {0}] score: {1:.5f}".format(k, fold_score))

print('Mean RF accuracy:', rf_accuracy / 5)


## What the comparison shows

The Random Forest's mean cross-validated accuracy came out higher than the single Decision Tree's. This lines up with the standard explanation for why ensembles of trees tend to generalize better: a single decision tree has low bias but high variance, it can fit training data very closely but is sensitive to small changes in the data. Averaging predictions across many trees, each trained on a bootstrapped sample with a random subset of features at each split, trades a small amount of bias for a meaningful reduction in variance.

Limitation: this notebook reports accuracy only. The breast cancer dataset is moderately imbalanced (roughly 63 percent benign, 37 percent malignant), so a precision/recall or confusion-matrix breakdown by class, particularly recall on the malignant class, would be the more clinically meaningful metric and is a natural next step. Neither model's hyperparameters were tuned; both ran on scikit-learn defaults.
